# Data Augmentation with Keyword Mapper

Create augmented variants of instruction-response pairs by replacing keywords with transliterated (translit) and native Arabic forms from the mapper dictionary.

## Section 1: Import Required Libraries

In [349]:
import os
import sys
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Set
import copy

# Set paths
data_dir = Path("C:\\Users\\user\\OneDrive\\Bureau\\TunisianDialogSystem\\llm\\sft\\data")
mapper_path = Path("C:\\Users\\user\\OneDrive\\Bureau\\TunisianDialogSystem\\llm\\sft\\data\\mapper.py")
cleaned_data_path = Path("C:\\Users\\user\\OneDrive\\Bureau\\TunisianDialogSystem\\llm\\sft\\data\\cleaned\\cleaned.csv")
augmented_dir = Path("C:\\Users\\user\\OneDrive\\Bureau\\TunisianDialogSystem\\llm\\sft\\data\\augmented")

# Create augmented directory if it doesn't exist
augmented_dir.mkdir(parents=True, exist_ok=True)

print("✅ Libraries imported successfully")
print(f"Data directory: {data_dir.resolve()}")
print(f"Mapper path: {mapper_path.resolve()}")
print(f"Cleaned data path: {cleaned_data_path.resolve()}")
print(f"Augmented output directory: {augmented_dir.resolve()}")

✅ Libraries imported successfully
Data directory: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data
Mapper path: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data\mapper.py
Cleaned data path: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data\cleaned\cleaned.csv
Augmented output directory: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data\augmented


## Section 2: Load the Keyword Mapper

In [350]:
# Add parent directory to path to import mapper
sys.path.insert(0, str(mapper_path.parent))
from mapper import KEYWORD_MAP

print(f"✅ Mapper loaded successfully")
print(f"Total keywords in mapper: {len(KEYWORD_MAP)}")
print(f"\n📋 Sample mapper entries (first 5):")
for i, (keyword, forms) in enumerate(list(KEYWORD_MAP.items())[:5]):
    print(f"  {i+1}. '{keyword}':")
    print(f"     - translit: {forms['translit']}")
    print(f"     - native: {forms['native']}")
    print(f"     - primary: {forms['primary']}")

# Create a set of all keywords (lowercase) for faster searching
mapper_keywords_lower = {key.lower(): key for key in KEYWORD_MAP.keys()}
print(f"\n✅ Mapper keywords ready for case-insensitive search")

✅ Mapper loaded successfully
Total keywords in mapper: 448

📋 Sample mapper entries (first 5):
  1. 'constipation':
     - translit: كونستيباسيون
     - native: إمساك
     - primary: إمساك
  2. 'acouphènes':
     - translit: أكوفان
     - native: طنين في الوذن
     - primary: طنين في الوذن
  3. 'palpitations':
     - translit: بالبيتاسيون
     - native: دقان قوي في القلب
     - primary: دقان قوي في القلب
  4. 'nez bouché':
     - translit: ني بوشي
     - native: خشم مسدود
     - primary: خشم مسدود
  5. 'essoufflement':
     - translit: إيسوفلمون
     - native: ضيق نفس
     - primary: ضيق نفس

✅ Mapper keywords ready for case-insensitive search


## Section 3: Load and Parse Clean Data

In [351]:
# Load cleaned data
try:
    df_clean = pd.read_csv(cleaned_data_path)
    print(f"✅ Cleaned data loaded successfully")
    print(f"   Shape: {df_clean.shape}")
    print(f"   Columns: {list(df_clean.columns)}")
    print(f"\n📊 First 3 rows:")
    print(df_clean.head(3))
    print(f"\n📈 Data info:")
    print(df_clean.info())
except Exception as e:
    print(f"❌ Error loading cleaned data: {e}")
    print(f"   Trying alternative paths...")
    
    # Try alternative paths
    alt_paths = [
        Path("./cleaned.csv"),
        Path("../cleaned.csv"),
        Path("../../cleaned.csv"),
    ]
    
    for alt_path in alt_paths:
        if alt_path.exists():
            print(f"   Found at: {alt_path.resolve()}")
            df_clean = pd.read_csv(alt_path)
            print(f"✅ Data loaded from: {alt_path}")
            break
    else:
        print("❌ Could not find cleaned data file in any location")

✅ Cleaned data loaded successfully
   Shape: (11857, 3)
   Columns: ['category', 'instruction', 'response']

📊 First 3 rows:
             category                                        instruction  \
0  cultural_knowledge  أعطيني لمحة على أشهر أنواع السلايط في الكوجينة...   
1        storytelling  كيفاش يعتبر تكريم كريم بقير وزهرة سليم في مجال...   
2  cultural_knowledge  كيفاش تنجم تونس تحافظ على التراث الفلكي الإسلا...   

                                            response  
0  الكوجينة التونسية معروفة بتنوع السلايط متاعها،...  
1  تكريم كريم بقير وزهرة سليم (مؤسسي InstaDeep) ي...  
2  تونس تنجم تواصل في الثنية هاذي عن طريق:\n• عمل...  

📈 Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11857 entries, 0 to 11856
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   category     11857 non-null  object
 1   instruction  11857 non-null  object
 2   response     11857 non-null  object
dtypes: object(3)
memor

## Section 4: Implement Augmentation Logic

In [352]:
def find_keywords_in_text(text: str, mapper_keywords_lower: Dict) -> List[Tuple[str, str]]:
    """
    Find all mapper keywords in text (case-insensitive).
    Returns list of (original_keyword, lowercase_key) tuples.
    """
    if not isinstance(text, str):
        return []
    
    found_keywords = []
    text_lower = text.lower()
    
    for keyword_lower, original_keyword in mapper_keywords_lower.items():
        # Use word boundary regex for more accurate matching
        pattern = r'\b' + re.escape(keyword_lower) + r'\b'
        if re.search(pattern, text_lower):
            found_keywords.append((original_keyword, keyword_lower))
    
    return found_keywords


def replace_keyword_in_text(text: str, keyword: str, replacement: str) -> str:
    """
    Replace keyword with replacement (case-insensitive) in text.
    Preserves case pattern when possible.
    """
    if not isinstance(text, str):
        return text
    
    # Case-insensitive replacement using word boundaries
    pattern = r'\b' + re.escape(keyword) + r'\b'
    return re.sub(pattern, replacement, text, flags=re.IGNORECASE)


def augment_entry(row: pd.Series, KEYWORD_MAP: Dict, mapper_keywords_lower: Dict) -> List[pd.Series]:
    """
    Create augmented variants for a single entry.
    Returns list of: [original_row, translit_variants..., native_variants...]
    """
    augmented_entries = [row.copy()]  # Keep original
    
    # Find keywords in both instruction and response
    instruction_keywords = find_keywords_in_text(row.get('instruction', ''), mapper_keywords_lower)
    response_keywords = find_keywords_in_text(row.get('response', ''), mapper_keywords_lower)
    
    all_keywords = set([kw[0] for kw in instruction_keywords + response_keywords])
    
    if not all_keywords:
        return augmented_entries
    
    # Create translit variant
    translit_row = row.copy()
    for keyword in all_keywords:
        if 'instruction' in translit_row.index:
            translit_row['instruction'] = replace_keyword_in_text(
                translit_row['instruction'], 
                keyword, 
                KEYWORD_MAP[keyword]['translit']
            )
        if 'response' in translit_row.index:
            translit_row['response'] = replace_keyword_in_text(
                translit_row['response'], 
                keyword, 
                KEYWORD_MAP[keyword]['translit']
            )
    
    translit_row['augmentation_type'] = 'translit'
    augmented_entries.append(translit_row)
    
    # Create native variant
    native_row = row.copy()
    for keyword in all_keywords:
        if 'instruction' in native_row.index:
            native_row['instruction'] = replace_keyword_in_text(
                native_row['instruction'], 
                keyword, 
                KEYWORD_MAP[keyword]['native']
            )
        if 'response' in native_row.index:
            native_row['response'] = replace_keyword_in_text(
                native_row['response'], 
                keyword, 
                KEYWORD_MAP[keyword]['native']
            )
    
    native_row['augmentation_type'] = 'native'
    augmented_entries.append(native_row)
    
    # Mark original row
    augmented_entries[0]['augmentation_type'] = 'original'
    
    return augmented_entries


def augment_question_words(row: pd.Series) -> List[pd.Series]:
    """
    Create variants by prepending 'تنجم تقلي' to instructions starting with question words.
    Question words: كيفاش، شنوة، علاش، شنوّة، شنية، أشنو
    Returns list of augmented variants.
    """
    QUESTION_WORDS = ['كيفاش', 'شنوة', 'علاش', 'شنوّة', 'شنية', 'أشنو', 'شنوا', 'شنو']
    PREPEND_TEXT = 'تنجم تقلي '
    
    augmented_entries = []
    instruction = row.get('instruction', '')
    
    if not isinstance(instruction, str) or not instruction.strip():
        return augmented_entries
    
    instr_lower = instruction.lower()
    
    # Check if instruction starts with any question word
    for question_word in QUESTION_WORDS:
        pattern = r'^' + re.escape(question_word)
        if re.search(pattern, instr_lower):
            # Create new variant with prepended text
            variant_row = row.copy()
            variant_row['instruction'] = PREPEND_TEXT + instruction
            variant_row['augmentation_type'] = 'question_variant'
            augmented_entries.append(variant_row)
            break  # Only add one variant per instruction
    
    return augmented_entries


def augment_parentheses_variants(row: pd.Series) -> List[pd.Series]:
    """
    Create variants from words with parentheses containing alternatives.
    Example: 'السكر (الحلويات)' creates a variant 'الحلويات'
    
    Strategy:
    1. Find all patterns: word (alternative)
    2. Create new variant with the alternative word
    3. Keep original but clean parentheses
    
    Returns list of: [original_cleaned, parentheses_variant1, parentheses_variant2, ...]
    """
    augmented_entries = []
    
    # Process both instruction and response
    for field in ['instruction', 'response']:
        text = row.get(field, '')
        if not isinstance(text, str) or not text.strip():
            continue
        
        # Find all patterns: word (alternative1 | alternative2)
        # Pattern: \S+\s*\([^)]+\)
        pattern = r'(\S+)\s*\(([^)]+)\)'
        matches = list(re.finditer(pattern, text))
        
        if not matches:
            continue
        
        # Create variants for each parentheses match
        for match in matches:
            main_word = match.group(1)
            alternatives_str = match.group(2)
            alternatives = [alt.strip() for alt in alternatives_str.split('|')]
            
            for alt in alternatives:
                if alt.strip():  # Non-empty alternative
                    # Create variant with this alternative instead
                    variant_row = row.copy()
                    
                    # Replace the pattern (main_word (alternatives)) with just the alternative
                    new_text = text[:match.start()] + alt + text[match.end():]
                    variant_row[field] = new_text
                    variant_row['augmentation_type'] = 'parentheses_variant'
                    augmented_entries.append(variant_row)
        
        # Create one variant with all parentheses cleaned (removed)
        if matches:
            cleaned_text = re.sub(pattern, r'\1', text)  # Replace with just the main word
            cleaned_row = row.copy()
            cleaned_row[field] = cleaned_text
            cleaned_row['augmentation_type'] = 'parentheses_cleaned'
            augmented_entries.append(cleaned_row)
    
    return augmented_entries


print("✅ Augmentation functions defined (mapper + question words + parentheses)")

✅ Augmentation functions defined (mapper + question words + parentheses)


## Section 5: Generate Augmented Variants

In [353]:
print("🚀 Starting augmentation process...")
print(f"Processing {len(df_clean)} entries...\n")

# Collect all augmented entries
augmented_list = []
entries_with_keywords = 0
entries_with_question_words = 0
entries_with_parentheses = 0
entries_without_augmentation = 0

for idx, row in df_clean.iterrows():
    # Strategy 1: Mapper-based augmentation (keyword replacement)
    augmented_variants = augment_entry(row, KEYWORD_MAP, mapper_keywords_lower)
    
    # Strategy 2: Question word variant augmentation
    question_variants = augment_question_words(row)
    
    # Strategy 3: Parentheses variant augmentation
    parentheses_variants = augment_parentheses_variants(row)
    
    # Track augmentation counts
    if len(augmented_variants) > 1:  # Has keyword augmentations
        entries_with_keywords += 1
    
    if len(question_variants) > 0:  # Has question word variant
        entries_with_question_words += 1
    
    if len(parentheses_variants) > 0:  # Has parentheses variants
        entries_with_parentheses += 1
    
    if len(augmented_variants) == 1 and len(question_variants) == 0 and len(parentheses_variants) == 0:
        entries_without_augmentation += 1
    
    # Combine all variants (original + keyword variants + question variants + parentheses variants)
    augmented_list.extend(augmented_variants)
    augmented_list.extend(question_variants)
    augmented_list.extend(parentheses_variants)
    
    if (idx + 1) % 100 == 0:
        print(f"✓ Processed {idx + 1} entries")

print(f"\n✅ Augmentation complete!")
print(f"   Original entries: {len(df_clean)}")
print(f"   Entries with keyword augmentations: {entries_with_keywords}")
print(f"   Entries with question word variants: {entries_with_question_words}")
print(f"   Entries with parentheses variants: {entries_with_parentheses}")
print(f"   Entries without augmentation: {entries_without_augmentation}")
print(f"   Total augmented entries (including originals): {len(augmented_list)}")
print(f"   Augmentation ratio: {len(augmented_list) / len(df_clean):.2f}x")

# Create augmented DataFrame
df_augmented = pd.DataFrame(augmented_list)
df_augmented = df_augmented.reset_index(drop=True)

print(f"\n📊 Augmented dataset shape: {df_augmented.shape}")
print(f"   Columns: {list(df_augmented.columns)}")
print(f"\n📋 Sample augmented entries (showing variants):")
sample_idx = 0
found_sample = False
for i in range(len(df_augmented)):
    if not found_sample and df_augmented.iloc[i].get('augmentation_type') == 'original':
        # Show original + next variants (up to 4 variants)
        sample_indices = [i]
        for j in range(i+1, min(i+5, len(df_augmented))):
            if df_augmented.iloc[j].get('augmentation_type') in ['translit', 'native', 'question_variant', 'parentheses_variant', 'parentheses_cleaned']:
                sample_indices.append(j)
        
        for sample_i, idx in enumerate(sample_indices[:4]):
            aug_type = df_augmented.iloc[idx].get('augmentation_type', 'unknown')
            print(f"\n   Index {idx} [{aug_type}]:")
            print(f"     Instruction: {df_augmented.iloc[idx].get('instruction', 'N/A')[:80]}...")
            print(f"     Response: {df_augmented.iloc[idx].get('response', 'N/A')[:80]}...")
        
        found_sample = True
        break

# Statistics on augmentation types
aug_stats = df_augmented['augmentation_type'].value_counts()
print(f"\n📈 Augmentation type distribution:")
for aug_type, count in aug_stats.items():
    print(f"   {aug_type}: {count} entries ({count/len(df_augmented)*100:.1f}%)")

🚀 Starting augmentation process...
Processing 11857 entries...

✓ Processed 100 entries
✓ Processed 200 entries
✓ Processed 300 entries
✓ Processed 400 entries
✓ Processed 500 entries
✓ Processed 600 entries
✓ Processed 700 entries
✓ Processed 800 entries
✓ Processed 900 entries
✓ Processed 1000 entries
✓ Processed 1100 entries
✓ Processed 1200 entries
✓ Processed 1300 entries
✓ Processed 1400 entries
✓ Processed 1500 entries
✓ Processed 1600 entries
✓ Processed 1700 entries
✓ Processed 1800 entries
✓ Processed 1900 entries
✓ Processed 2000 entries
✓ Processed 2100 entries
✓ Processed 2200 entries
✓ Processed 2300 entries


KeyboardInterrupt: 

## Section 6: Save Augmented Data

In [ ]:
# Save augmented data
print("💾 Saving augmented data...\n")

# Main augmented CSV
augmented_csv_path = augmented_dir / "augmented_all.csv"
df_augmented.to_csv(augmented_csv_path, index=False)
print(f"✅ Saved: {augmented_csv_path}")
print(f"   Shape: {df_augmented.shape}")

# Save by augmentation type
for aug_type in ['original', 'translit', 'native', 'question_variant', 'parentheses_variant', 'parentheses_cleaned']:
    df_type = df_augmented[df_augmented['augmentation_type'] == aug_type]
    if len(df_type) > 0:
        type_path = augmented_dir / f"augmented_{aug_type}.csv"
        df_type.to_csv(type_path, index=False)
        print(f"✅ Saved: {type_path}")
        print(f"   Shape: {df_type.shape}")

# Save summary JSON
summary = {
    "original_entries": len(df_clean),
    "augmented_entries": len(df_augmented),
    "augmentation_ratio": len(df_augmented) / len(df_clean),
    "entries_with_keyword_augmentations": entries_with_keywords,
    "entries_with_question_variants": entries_with_question_words,
    "entries_with_parentheses_variants": entries_with_parentheses,
    "entries_without_augmentation": entries_without_augmentation,
    "augmentation_types": {
        "original": int(len(df_augmented[df_augmented['augmentation_type'] == 'original'])),
        "translit": int(len(df_augmented[df_augmented['augmentation_type'] == 'translit'])),
        "native": int(len(df_augmented[df_augmented['augmentation_type'] == 'native'])),
        "question_variant": int(len(df_augmented[df_augmented['augmentation_type'] == 'question_variant'])),
        "parentheses_variant": int(len(df_augmented[df_augmented['augmentation_type'] == 'parentheses_variant'])),
        "parentheses_cleaned": int(len(df_augmented[df_augmented['augmentation_type'] == 'parentheses_cleaned'])),
    },
    "total_keywords_in_mapper": len(KEYWORD_MAP),
    "question_words_tracked": ['كيفاش', 'شنوة', 'علاش', 'شنوّة', 'شنية', 'أشنو', 'شنوا', 'شنو'],
    "prepended_text_for_questions": 'تنجم تقلي',
    "parentheses_augmentation_note": "Creates variants from text patterns like 'word (alternative)' by extracting alternatives",
}

summary_path = augmented_dir / "augmentation_summary.json"
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n✅ Saved: {summary_path}")
print(f"\n📊 Augmentation Summary:")
print(json.dumps(summary, ensure_ascii=False, indent=2))

# Also save merged version (all + cleaned data)
merged_path = augmented_dir / "augmented_all_merged.csv"
df_merged = pd.concat([df_clean, df_augmented], ignore_index=True)
df_merged.to_csv(merged_path, index=False)
print(f"\n✅ Saved merged (original + augmented): {merged_path}")
print(f"   Shape: {df_merged.shape}")

print(f"\n🎉 Augmentation process complete!")
print(f"   Output directory: {augmented_dir.resolve()}")

💾 Saving augmented data...

✅ Saved: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data\augmented\augmented_all.csv
   Shape: (13947, 4)
✅ Saved: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data\augmented\augmented_original.csv
   Shape: (1045, 4)
✅ Saved: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data\augmented\augmented_translit.csv
   Shape: (1045, 4)
✅ Saved: C:\Users\user\OneDrive\Bureau\TunisianDialogSystem\llm\sft\data\augmented\augmented_native.csv
   Shape: (1045, 4)


NameError: name 'entries_with_question_words' is not defined

## Section 7: Quality Control & Analysis

In [ ]:
print("🔍 Quality Control Checks\n")

# Check 1: No empty instructions/responses
empty_instr = df_augmented['instruction'].isna().sum() + (df_augmented['instruction'] == '').sum()
empty_resp = df_augmented['response'].isna().sum() + (df_augmented['response'] == '').sum()
print(f"✅ Check 1 - Empty values:")
print(f"   Empty instructions: {empty_instr}")
print(f"   Empty responses: {empty_resp}")

# Check 2: Augmentation type consistency
print(f"\n✅ Check 2 - Augmentation type consistency:")
print(f"   All entries have 'augmentation_type': {('augmentation_type' in df_augmented.columns) and (df_augmented['augmentation_type'].isna().sum() == 0)}")

# Check 3: Sample differences between variants
print(f"\n✅ Check 3 - Sample variant comparisons:")

# Show mapper variants
print(f"\n   📍 Mapper-based variants (keyword replacement):")
sample_df = df_augmented[df_augmented['augmentation_type'] == 'original'].head(1)
for orig_idx in sample_df.index:
    orig_row = df_augmented.iloc[orig_idx]
    orig_instr = orig_row.get('instruction', '')
    
    # Find corresponding translit variant
    translit_mask = (df_augmented.index > orig_idx) & (df_augmented['augmentation_type'] == 'translit')
    if translit_mask.any():
        next_idx = df_augmented.index[translit_mask][0]
        translit_row = df_augmented.iloc[next_idx]
        translit_instr = translit_row.get('instruction', '')
        
        if orig_instr != translit_instr:
            print(f"\n      Variant {orig_idx}→{next_idx}:")
            print(f"        Original: {orig_instr[:70]}...")
            print(f"        Translit: {translit_instr[:70]}...")

# Show question word variants
print(f"\n   📍 Question word variants (prepended 'تنجم تقلي'):")
question_variants = df_augmented[df_augmented['augmentation_type'] == 'question_variant'].head(2)
if len(question_variants) > 0:
    for idx, qvar_row in question_variants.iterrows():
        instruction = qvar_row.get('instruction', '')
        if instruction.startswith('تنجم تقلي'):
            original_instr = instruction.replace('تنجم تقلي ', '', 1)
            print(f"\n      Index {idx}:")
            print(f"        Original: {original_instr[:70]}...")
            print(f"        Variant:  {instruction[:70]}...")
else:
    print("      (No question word variants found)")

# Show parentheses variants
print(f"\n   📍 Parentheses variants (word replacements):")
paren_variants = df_augmented[df_augmented['augmentation_type'] == 'parentheses_variant'].head(2)
if len(paren_variants) > 0:
    for idx, pvar_row in paren_variants.iterrows():
        instr = pvar_row.get('instruction', '')
        print(f"\n      Index {idx}:")
        print(f"        Instruction: {instr[:80]}...")
else:
    print("      (No parentheses variants found)")

paren_cleaned = df_augmented[df_augmented['augmentation_type'] == 'parentheses_cleaned'].head(1)
if len(paren_cleaned) > 0:
    for idx, pclean_row in paren_cleaned.iterrows():
        instr = pclean_row.get('instruction', '')
        print(f"\n   📍 Parentheses cleaned (removed):")
        print(f"      Index {idx}: {instr[:80]}...")

# Check 4: Keyword coverage analysis
print(f"\n✅ Check 4 - Keyword coverage:")
keywords_found = set()
for idx, row in df_augmented.iterrows():
    instruction = row.get('instruction', '')
    response = row.get('response', '')
    combined = str(instruction) + ' ' + str(response)
    
    for keyword in mapper_keywords_lower.values():
        if keyword.lower() in combined.lower():
            keywords_found.add(keyword)

print(f"   Unique keywords found in augmented data: {len(keywords_found)}")
print(f"   Total keywords in mapper: {len(KEYWORD_MAP)}")
print(f"   Coverage: {len(keywords_found)/len(KEYWORD_MAP)*100:.1f}%")

# Check 5: Question word coverage
print(f"\n✅ Check 5 - Question word variant coverage:")
question_word_count = len(df_augmented[df_augmented['augmentation_type'] == 'question_variant'])
print(f"   Question word variants created: {question_word_count}")
print(f"   Expected (entries starting with question words): {entries_with_question_words}")

# Check 6: Parentheses variant coverage
print(f"\n✅ Check 6 - Parentheses variant coverage:")
paren_var_count = len(df_augmented[df_augmented['augmentation_type'] == 'parentheses_variant'])
paren_clean_count = len(df_augmented[df_augmented['augmentation_type'] == 'parentheses_cleaned'])
print(f"   Parentheses variants created: {paren_var_count}")
print(f"   Parentheses cleaned created: {paren_clean_count}")
print(f"   Total parentheses augmentations: {paren_var_count + paren_clean_count}")
print(f"   Entries with parentheses: {entries_with_parentheses}")

if keywords_found:
    print(f"\n   Sample keywords found:")
    for keyword in list(keywords_found)[:10]:
        print(f"     - {keyword}")

print(f"\n✅ All quality checks complete!")